# colab_runner - Orchestratore Colab per ProjectML

Notebook di setup per eseguire il progetto *Waste Type Identification* su Google Colab e salvare automaticamente i risultati su GitHub.

Cosa fa:

1. Monta Google Drive.
2. Clona o aggiorna `fabriziodrr/ProjectML`.
3. Copia ed estrae il dataset sul disco locale di Colab.
4. Collega il dataset dentro la repo con un symlink.
5. Configura `sys.path`.
6. Esegue i notebook di esperimento indicati nella configurazione.
7. Fa commit e push delle modifiche su GitHub.

Prima di eseguirlo, controlla la cella di configurazione e modifica i path marcati come azionabili.

## 1. Mount di Google Drive

Serve per leggere `dataset.zip` e, se vuoi, per mantenere copie persistenti di checkpoint o artefatti.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configurazione

Modifica qui i percorsi del dataset, i notebook da eseguire e l'identita git.

In [ ]:
import os

REPO_OWNER = 'fabriziodrr'
REPO_NAME = 'ProjectML'
BRANCH = 'main'
REPO_PATH = f'/content/{REPO_NAME}'
PUBLIC_REPO_URL = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'

# AZIONABILE: cambia questo path in base a dove hai caricato lo zip sul tuo Drive.
DRIVE_ZIP_PATH = '/content/drive/MyDrive/dataset.zip'
LOCAL_ZIP_PATH = '/content/dataset.zip'
EXTRACT_DIR = '/content'

# AZIONABILE: nome della cartella prodotta dall'estrazione dello zip.
# Esempi possibili: 'dataset', 'waste_type_identification'.
DATASET_DIR_NAME = 'dataset'
LOCAL_DATASET_PATH = os.path.join(EXTRACT_DIR, DATASET_DIR_NAME)
REPO_DATASET_PATH = os.path.join(REPO_PATH, 'dataset')

# AZIONABILE: inserisci qui i notebook di esperimento da lanciare, relativi alla root della repo.
# Lascia la lista vuota se vuoi solo preparare ambiente e fare push manuale.
EXPERIMENT_NOTEBOOKS = [
    # 'src/create_splits.ipynb',
    # 'src/e1/e1_baseline.ipynb',
]

# AZIONABILE: identita usata nei commit generati da Colab.
GIT_USER_NAME = 'fabriziodrr'
GIT_USER_EMAIL = ''  # esempio: 'nome@example.com'

print('Repo:', PUBLIC_REPO_URL)
print('Repo path:', REPO_PATH)
print('Dataset zip:', DRIVE_ZIP_PATH)
print('Dataset locale atteso:', LOCAL_DATASET_PATH)

## 3. Clone o update della repo

Chiede un GitHub token personale. Il token viene usato per clone/pull/push e il remote viene riportato all'URL pubblico alla fine delle operazioni, per non salvarlo nella configurazione git.

In [ ]:
import os
import subprocess
from getpass import getpass

def run(cmd, cwd=None, check=True):
    print('$', ' '.join(cmd))
    return subprocess.run(cmd, cwd=cwd, check=check)

GITHUB_TOKEN = getpass('GitHub token personale con permessi repo: ')
AUTH_REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'

if not os.path.exists(REPO_PATH):
    print('Repo non presente nel runtime: avvio clone.')
    run(['git', 'clone', '--branch', BRANCH, AUTH_REPO_URL, REPO_PATH])
else:
    print('Repo gia presente nel runtime: aggiorno da GitHub.')

run(['git', 'remote', 'set-url', 'origin', AUTH_REPO_URL], cwd=REPO_PATH)
run(['git', 'pull', '--rebase', 'origin', BRANCH], cwd=REPO_PATH)
run(['git', 'remote', 'set-url', 'origin', PUBLIC_REPO_URL], cwd=REPO_PATH)

os.chdir(REPO_PATH)
print('Working directory:', os.getcwd())

## 4. Preparazione dataset su SSD locale

Copia lo zip da Drive a `/content`, poi lo estrae. Leggere immagini dal disco locale di Colab e molto piu veloce che leggerle direttamente da Drive.

In [ ]:
import os
import shutil
import subprocess

if not os.path.exists(LOCAL_DATASET_PATH):
    if not os.path.exists(DRIVE_ZIP_PATH):
        raise FileNotFoundError(f'Dataset zip non trovato: {DRIVE_ZIP_PATH}')

    print('Copio dataset zip da Drive al disco locale Colab...')
    shutil.copy2(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)

    print('Estraggo dataset...')
    subprocess.run(['unzip', '-q', LOCAL_ZIP_PATH, '-d', EXTRACT_DIR], check=True)
else:
    print('Dataset gia presente localmente.')

if not os.path.exists(LOCAL_DATASET_PATH):
    print('Cartelle presenti in /content:')
    print(os.listdir('/content'))
    raise FileNotFoundError(
        f'La cartella dataset attesa non esiste: {LOCAL_DATASET_PATH}. '
        'Aggiorna DATASET_DIR_NAME nella cella di configurazione.'
    )

print('Dataset locale pronto:', LOCAL_DATASET_PATH)

## 5. Symlink dataset nella repo

Crea `ProjectML/dataset` come collegamento alla cartella dataset estratta in `/content`. In questo modo il codice puo usare path relativi alla repo senza copiare migliaia di immagini.

In [ ]:
import os
import shutil

if os.path.islink(REPO_DATASET_PATH) or os.path.isfile(REPO_DATASET_PATH):
    os.unlink(REPO_DATASET_PATH)
elif os.path.isdir(REPO_DATASET_PATH):
    shutil.rmtree(REPO_DATASET_PATH)

os.symlink(LOCAL_DATASET_PATH, REPO_DATASET_PATH)
print('Symlink creato:', REPO_DATASET_PATH, '->', LOCAL_DATASET_PATH)
print('Contenuto dataset:')
print(os.listdir(REPO_DATASET_PATH)[:30])

## 6. Configurazione Python path

Aggiunge root repo e, se presente, `src/` a `sys.path`.

In [ ]:
import os
import sys

for path in [REPO_PATH, os.path.join(REPO_PATH, 'src')]:
    if os.path.isdir(path) and path not in sys.path:
        sys.path.append(path)
        print('Aggiunto a sys.path:', path)

print('File nella repo:')
print(os.listdir(REPO_PATH))

## 7. Esecuzione esperimenti

Esegue in ordine i notebook elencati in `EXPERIMENT_NOTEBOOKS`. Se la lista e vuota, questa cella non fa nulla.

In [ ]:
ipython = get_ipython()

for notebook in EXPERIMENT_NOTEBOOKS:
    notebook_path = os.path.join(REPO_PATH, notebook)
    if not os.path.exists(notebook_path):
        raise FileNotFoundError(f'Notebook non trovato: {notebook_path}')
    print('Eseguo:', notebook_path)
    ipython.run_line_magic('run', f'"{notebook_path}"')

## 8. Commit e push su GitHub

Aggiunge tutte le modifiche, crea un commit solo se ci sono cambiamenti e fa push su `main`. Il token non viene lasciato nel remote git.

In [ ]:
import datetime
import os
import subprocess
from getpass import getpass

os.chdir(REPO_PATH)

if GIT_USER_NAME:
    run(['git', 'config', 'user.name', GIT_USER_NAME], cwd=REPO_PATH)
if GIT_USER_EMAIL:
    run(['git', 'config', 'user.email', GIT_USER_EMAIL], cwd=REPO_PATH)

run(['git', 'add', '-A'], cwd=REPO_PATH)
status = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_PATH).decode().strip()

if not status:
    print('Nessuna modifica da committare.')
else:
    commit_message = 'Aggiornamento da Colab ' + datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    run(['git', 'commit', '-m', commit_message], cwd=REPO_PATH)

    token = globals().get('GITHUB_TOKEN')
    if not token:
        token = getpass('GitHub token personale con permessi repo: ')

    auth_url = f'https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
    try:
        run(['git', 'remote', 'set-url', 'origin', auth_url], cwd=REPO_PATH)
        run(['git', 'pull', '--rebase', 'origin', BRANCH], cwd=REPO_PATH)
        run(['git', 'push', 'origin', BRANCH], cwd=REPO_PATH)
    finally:
        run(['git', 'remote', 'set-url', 'origin', PUBLIC_REPO_URL], cwd=REPO_PATH, check=False)

print('Operazione completata.')